
# Chapter 13: Machine learning techniques


### Importing the necessary packages

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
import random as rd
rd.seed(0)

### Download and unzip dataset

In [4]:
#!/bin/bash
!curl -L -o /content/titanic-dataset.zip\
  https://www.kaggle.com/api/v1/datasets/download/yasserh/titanic-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 22564  100 22564    0     0  51560      0 --:--:-- --:--:-- --:--:--     0


In [5]:
# Unzip dataset
#!/bin/bash
!unzip /content/titanic-dataset.zip -d /content/

Archive:  /content/titanic-dataset.zip
  inflating: /content/Titanic-Dataset.csv  


## 13.1 Loading and exploring the dataset

In [6]:
# IMPORTANT: ONLY RUN THIS CELL IF YOU ARE WORKING ON A COLAB

url = "https://raw.githubusercontent.com/luisguiserrano/manning/master/Chapter_13_End_to_end_example/titanic.csv"
raw_data = pd.read_csv(url)
raw_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Next, we can explore the dataset.

In [7]:
# Examining the length of the dataset
print("The dataset has", len(raw_data), "rows")

The dataset has 891 rows


In [8]:
# Examining the columns in the dataset
print("Columns (features of the dataset)")
raw_data.columns

Columns (features of the dataset)


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

Examining the labels

In [9]:
# Examining the labels
print("Labels")
raw_data["Survived"]

Labels


,Survived
0,0
1,1
2,1
3,1
4,0
...,...
886,0
887,1
888,0
889,1


Examining how many passengers survived

In [10]:
# Examining how many passengers survived
print(sum(raw_data['Survived']), 'passengers survived out of', len(raw_data))

342 passengers survived out of 891


In [11]:
# One can look at several columns together
raw_data[["Name", "Age"]]

,Name,Age
0,"Braund, Mr. Owen Harris",22.0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
2,"Heikkinen, Miss. Laina",26.0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0
4,"Allen, Mr. William Henry",35.0
...,...,...
886,"Montvila, Rev. Juozas",27.0
887,"Graham, Miss. Margaret Edith",19.0
888,"Johnston, Miss. Catherine Helen ""Carrie""",NaN
889,"Behr, Mr. Karl Howell",26.0


## 13.2. Cleaning up the data

### Now, let's look at how many columns have missing data

In [12]:
raw_data.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


The Cabin column is missing too many values to be useful. Let's drop it altogether.

In [13]:
raw_data['Cabin']

,Cabin
0,NaN
1,C85
2,NaN
3,C123
4,NaN
...,...
886,NaN
887,B42
888,NaN
889,C148


In [14]:
print("The Cabin column is missing", sum(raw_data['Cabin'].isna()), "values out of",len(raw_data['Cabin']))

The Cabin column is missing 687 values out of 891


### Exemine dataset after dropping Cabin feature (column)

In [15]:
preprocessed_data = raw_data.drop('Cabin', axis=1)

In [16]:
preprocessed_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


### Filling in missing data

Other columns such as Age or Embarked are missing some values, but they can still be useful.



In [17]:
preprocessed_data['Age']

,Age
0,22.0
1,38.0
2,26.0
3,35.0
4,35.0
...,...
886,27.0
887,19.0
888,NaN
889,26.0


#### For the age column, let's fill in the missing values with the median of all ages.

In [18]:
median_age = raw_data["Age"].median()
median_age

28.0

In [19]:
preprocessed_data["Age"] = preprocessed_data["Age"].fillna(median_age)

In [20]:
rows_with_age_median = preprocessed_data[preprocessed_data['Age'] == median_age]
print(len(rows_with_age_median))

202


In [21]:
preprocessed_data.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


#### For the Embarked column, let's make a new category called 'U', for Unknown port of embarkment.

In [22]:
preprocessed_data["Embarked"] = preprocessed_data["Embarked"].fillna('U')

In [23]:
rows_with_embarked_u = preprocessed_data[preprocessed_data['Embarked'] == 'U']
rows_with_embarked_u

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,U
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,U


In [24]:
preprocessed_data.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [25]:
preprocessed_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


### Save cleaned dataset for later use

In [26]:
preprocessed_data.to_csv('./clean_titanic_data.csv', index=None)

## 13.3 Manipulating the features

### Load cleaned dataset we stored in cleaning section

In [27]:
preprocessed_data = pd.read_csv('./clean_titanic_data.csv')
preprocessed_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


* One-hot encoding
* Binning
* Feature selection

## 13.3.1 One-hot encoding

#### One-hot encoding the gender feature

In [28]:
gender_columns = pd.get_dummies(preprocessed_data['Sex'], prefix='Sex')
print(gender_columns)
embarked_columns = pd.get_dummies(preprocessed_data["Embarked"], prefix="Embarked")
print(embarked_columns)

     Sex_female  Sex_male
0         False      True
1          True     False
2          True     False
3          True     False
4         False      True
..          ...       ...
886       False      True
887        True     False
888        True     False
889       False      True
890       False      True

[891 rows x 2 columns]
     Embarked_C  Embarked_Q  Embarked_S  Embarked_U
0         False       False        True       False
1          True       False       False       False
2         False       False        True       False
3         False       False        True       False
4         False       False        True       False
..          ...         ...         ...         ...
886       False       False        True       False
887       False       False        True       False
888       False       False        True       False
889        True       False       False       False
890       False        True       False       False

[891 rows x 4 columns]


#### Add One-hot encoded columns to data

In [29]:
preprocessed_data = pd.concat([preprocessed_data, gender_columns], axis=1)
preprocessed_data = pd.concat([preprocessed_data, embarked_columns], axis=1)

#### Remove Original columns (features)

In [30]:
preprocessed_data = preprocessed_data.drop(['Sex', 'Embarked'], axis=1)

In [31]:
preprocessed_data.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,False,True,False,False,True,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,True,False,True,False,False,False
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,True,False,False,False,True,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,True,False,False,False,True,False
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,False,True,False,False,True,False


### A rule of thumb for when to one-hot encode or not

In [32]:
class_survived = preprocessed_data[['Pclass', 'Survived']]

first_class = class_survived[class_survived['Pclass'] == 1]
second_class = class_survived[class_survived['Pclass'] == 2]
third_class = class_survived[class_survived['Pclass'] == 3]

print("In first class", sum(first_class['Survived'])/len(first_class)*100, "% of passengers survived")
print("In second class", sum(second_class['Survived'])/len(second_class)*100, "% of passengers survived")
print("In third class", sum(third_class['Survived'])/len(third_class)*100, "% of passengers survived")

In first class 62.96296296296296 % of passengers survived
In second class 47.28260869565217 % of passengers survived
In third class 24.236252545824847 % of passengers survived


In [33]:
categorized_pclass_columns = pd.get_dummies(preprocessed_data['Pclass'], prefix='Pclass')
preprocessed_data = pd.concat([preprocessed_data, categorized_pclass_columns], axis=1)
preprocessed_data = preprocessed_data.drop(['Pclass'], axis=1)

In [34]:
preprocessed_data.head()

,PassengerId,Survived,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3
0,1,0,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,False,True,False,False,True,False,False,False,True
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,True,False,True,False,False,False,True,False,False
2,3,1,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,True,False,False,False,True,False,False,False,True
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,True,False,False,False,True,False,True,False,False
4,5,0,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,False,True,False,False,True,False,False,False,True


## 13.3.3 Binning

### Split age feature into several different buckets

In [35]:
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
categorized_age = pd.cut(preprocessed_data['Age'], bins)
preprocessed_data['Categorized_age'] = categorized_age
preprocessed_data = preprocessed_data.drop(["Age"], axis=1)

In [36]:
preprocessed_data.head()

,PassengerId,Survived,Name,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3,Categorized_age
0,1,0,"Braund, Mr. Owen Harris",1,0,A/5 21171,7.2500,False,True,False,False,True,False,False,False,True,"(20, 30]"
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,PC 17599,71.2833,True,False,True,False,False,False,True,False,False,"(30, 40]"
2,3,1,"Heikkinen, Miss. Laina",0,0,STON/O2. 3101282,7.9250,True,False,False,False,True,False,False,False,True,"(20, 30]"
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,113803,53.1000,True,False,False,False,True,False,True,False,False,"(30, 40]"
4,5,0,"Allen, Mr. William Henry",0,0,373450,8.0500,False,True,False,False,True,False,False,False,True,"(30, 40]"


### Turn binned numerical feature (age) into categorical with One-hot endcoding

In [37]:
cagegorized_age_columns = pd.get_dummies(preprocessed_data['Categorized_age'], prefix='Categorized_age')
preprocessed_data = pd.concat([preprocessed_data, cagegorized_age_columns], axis=1)
preprocessed_data = preprocessed_data.drop(['Categorized_age'], axis=1)

In [38]:
preprocessed_data.head()

,PassengerId,Survived,Name,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,...,Pclass_2,Pclass_3,"Categorized_age_(0, 10]","Categorized_age_(10, 20]","Categorized_age_(20, 30]","Categorized_age_(30, 40]","Categorized_age_(40, 50]","Categorized_age_(50, 60]","Categorized_age_(60, 70]","Categorized_age_(70, 80]"
0,1,0,"Braund, Mr. Owen Harris",1,0,A/5 21171,7.2500,False,True,False,...,False,True,False,False,True,False,False,False,False,False
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,PC 17599,71.2833,True,False,True,...,False,False,False,False,False,True,False,False,False,False
2,3,1,"Heikkinen, Miss. Laina",0,0,STON/O2. 3101282,7.9250,True,False,False,...,False,True,False,False,True,False,False,False,False,False
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,113803,53.1000,True,False,False,...,False,False,False,False,False,True,False,False,False,False
4,5,0,"Allen, Mr. William Henry",0,0,373450,8.0500,False,True,False,...,False,True,False,False,False,True,False,False,False,False


## 13.3.3 Feature selection

In [39]:
preprocessed_data = preprocessed_data.drop(['Name', 'Ticket', 'PassengerId'], axis=1)

In [62]:
preprocessed_data.head()

,Survived,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,...,Pclass_2,Pclass_3,"Categorized_age_(0, 10]","Categorized_age_(10, 20]","Categorized_age_(20, 30]","Categorized_age_(30, 40]","Categorized_age_(40, 50]","Categorized_age_(50, 60]","Categorized_age_(60, 70]","Categorized_age_(70, 80]"
0,0,1,0,7.2500,False,True,False,False,True,False,...,False,True,False,False,True,False,False,False,False,False
1,1,1,0,71.2833,True,False,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,1,0,0,7.9250,True,False,False,False,True,False,...,False,True,False,False,True,False,False,False,False,False
3,1,1,0,53.1000,True,False,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
4,0,0,0,8.0500,False,True,False,False,True,False,...,False,True,False,False,False,True,False,False,False,False


### Save cleaned dataset for later use

In [41]:
preprocessed_data.to_csv('./preprocessed_titanic_data.csv', index=None)

## 13.4 Training models

### Load pre-processed dataset we stored in previous sections

In [42]:
data = pd.read_csv('./preprocessed_titanic_data.csv')
data.head()

,Survived,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,...,Pclass_2,Pclass_3,"Categorized_age_(0, 10]","Categorized_age_(10, 20]","Categorized_age_(20, 30]","Categorized_age_(30, 40]","Categorized_age_(40, 50]","Categorized_age_(50, 60]","Categorized_age_(60, 70]","Categorized_age_(70, 80]"
0,0,1,0,7.2500,False,True,False,False,True,False,...,False,True,False,False,True,False,False,False,False,False
1,1,1,0,71.2833,True,False,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2,1,0,0,7.9250,True,False,False,False,True,False,...,False,True,False,False,True,False,False,False,False,False
3,1,1,0,53.1000,True,False,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
4,0,0,0,8.0500,False,True,False,False,True,False,...,False,True,False,False,False,True,False,False,False,False


### 13.4.1 Features-labels split and train-validation split

#### Features-labels split

In [43]:
features = data.drop(["Survived"], axis=1)
labels = data["Survived"]

In [44]:
features.head()

,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3,"Categorized_age_(0, 10]","Categorized_age_(10, 20]","Categorized_age_(20, 30]","Categorized_age_(30, 40]","Categorized_age_(40, 50]","Categorized_age_(50, 60]","Categorized_age_(60, 70]","Categorized_age_(70, 80]"
0,1,0,7.2500,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False
1,1,0,71.2833,True,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False
2,0,0,7.9250,True,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False
3,1,0,53.1000,True,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False
4,0,0,8.0500,False,True,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False


In [45]:
labels

,Survived
0,0
1,1
2,1
3,1
4,0
...,...
886,0
887,1
888,0
889,1


#### Train-validation split

In [46]:
from sklearn.model_selection import train_test_split

In [47]:
# remark: we fix random_state the end, to make sure we always get the same split
features_train, features_validation_test, labels_train, labels_validation_test = train_test_split(
    features, labels, test_size=0.4, random_state=100)

In [48]:
features_validation, features_test, labels_validation, labels_test = train_test_split(
    features_validation_test, labels_validation_test, test_size=0.5, random_state=100)

In [49]:
print(len(features_train))
print(len(features_validation))
print(len(features_test))
print(len(labels_train))
print(len(labels_validation))
print(len(labels_test))

534
178
179
534
178
179


### 13.4.2 Training different models on our dataset

We'll train four models with basic model:

* Logistic regression (perceptron)
* Decision tree
* Naive Bayes
* Support vector machine (SVM)

#### Logistic regression (perceptron)

In [63]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression()
lr_model.fit(features_train, labels_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

#### Decision tree

In [64]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier()
dt_model.fit(features_train, labels_train)

DecisionTreeClassifier()

#### Naive Bayes - Guassian Naive Bayes model

In [65]:
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(features_train, labels_train)

GaussianNB()

#### Support vector machine (SVM)

In [66]:
from sklearn.svm import SVC

svm_model = SVC()
svm_model.fit(features_train, labels_train)

SVC()



---



We'll train three models with Ensemble techniques:

* Random forest
* Gradient boosted tree
* AdaBoost

#### Random forest (with Decision tree weak learners)

In [67]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier()
rf_model.fit(features_train, labels_train)

RandomForestClassifier()

#### Gradient boosted tree

In [68]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier()
gb_model.fit(features_train, labels_train)

GradientBoostingClassifier()

#### AdaBoost

In [69]:
from sklearn.ensemble import AdaBoostClassifier

ab_model = AdaBoostClassifier()
ab_model.fit(features_train, labels_train)

AdaBoostClassifier()

#### XGboost

Rename feature (column) names for XGboost model

In [95]:
import re

def clean_col(col: str) -> str:
    col = str(col)
    # remove forbidden chars and parentheses, make it nice
    col = re.sub(r"[\[\]<>\(\)]", "", col)
    col = col.replace(" ", "_")
    return col

features_train_xgboost = features_train.rename(columns=clean_col)
features_validation_xgboost = features_validation.rename(columns=clean_col)
features_test_xgboost = features_test.rename(columns=clean_col)

In [85]:
from xgboost import XGBClassifier

xgboost_model = XGBClassifier()
xgboost_model.fit(features_train_xgboost, labels_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

#### Building and training the neural network

In [107]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# X: features_train (DataFrame), y: labels_train (0/1)
X_train = features_train.values.astype("float32")
X_val   = features_validation.values.astype("float32")
X_test  = features_test.values.astype("float32")

y_train = labels_train.astype("float32").values
y_val   = labels_validation.astype("float32").values
y_test  = labels_test.astype("float32").values

model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.1))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1, activation='sigmoid'))  # single neuron for binary classification

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,457 (13.50 KB)

 Trainable params: 3,457 (13.50 KB)

 Non-trainable params: 0 (0.00 B)

In [108]:
# Neural Network training
nn_model = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

Epoch 1/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 134ms/step - accuracy: 0.5343 - loss: 1.0381 - val_accuracy: 0.6348 - val_loss: 0.6718
Epoch 2/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6046 - loss: 1.0335 - val_accuracy: 0.7022 - val_loss: 0.6025
Epoch 3/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6465 - loss: 0.8149 - val_accuracy: 0.7135 - val_loss: 0.5690
Epoch 4/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6487 - loss: 0.9204 - val_accuracy: 0.7247 - val_loss: 0.5930
Epoch 5/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6864 - loss: 0.7118 - val_accuracy: 0.6742 - val_loss: 0.5900
Epoch 6/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6863 - loss: 0.6752 - val_accuracy: 0.7135 - val_loss: 0.5485
Epoch 7/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7244 - loss: 0.6106 - val_accuracy: 0.7247 - val_loss: 0.5391
Epoch 8/50
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6974 - loss: 0.7391 - val_accuracy: 0.7303 - val_los

### 13.4.3 Evaluating the models

#### Accuracy

In [110]:
print("Scores of the models")
print("Logistic regression:", lr_model.score(features_validation, labels_validation))
print("Decision tree:", dt_model.score(features_validation, labels_validation))
print("Naive Bayes:", nb_model.score(features_validation, labels_validation))
print("SVM:", svm_model.score(features_validation, labels_validation))
print("Random forest:", rf_model.score(features_validation, labels_validation))
print("Gradient boosting:", gb_model.score(features_validation, labels_validation))
print("AdaBoost:", ab_model.score(features_validation, labels_validation))
print("XGBoost:", xgboost_model.score(features_validation_xgboost, labels_validation))

loss, accuracy = model.evaluate(X_test, y_test)
print("Neural Network accuracy:", accuracy)

Scores of the models
Logistic regression: 0.7696629213483146
Decision tree: 0.7640449438202247
Naive Bayes: 0.7471910112359551
SVM: 0.6797752808988764
Random forest: 0.7752808988764045
Gradient boosting: 0.8089887640449438
AdaBoost: 0.7359550561797753
XGBoost: 0.7808988764044944
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.8024 - loss: 0.4608
Neural Network accuracy: 0.7932960987091064


#### Recall

In [111]:
from sklearn.metrics import recall_score

print("Recall of the models:")

lr_predicted_labels = lr_model.predict(features_validation)
print("Logistic regression:", recall_score(labels_validation, lr_predicted_labels))

dt_predicted_labels = dt_model.predict(features_validation)
print("Decision Tree:", recall_score(labels_validation, dt_predicted_labels))

nb_predicted_labels = nb_model.predict(features_validation)
print("Naive Bayes:", recall_score(labels_validation, nb_predicted_labels))

svm_predicted_labels = svm_model.predict(features_validation)
print("Support Vector Machine:", recall_score(labels_validation, svm_predicted_labels))

rf_predicted_labels = rf_model.predict(features_validation)
print("Random Forest:", recall_score(labels_validation, rf_predicted_labels))

gb_predicted_labels = gb_model.predict(features_validation)
print("Gradient boosting:", recall_score(labels_validation, gb_predicted_labels))

ab_predicted_labels = ab_model.predict(features_validation)
print("AdaBoost:", recall_score(labels_validation, ab_predicted_labels))

xgb_predicted_labels = xgboost_model.predict(features_validation_xgboost)
print("XGBoost:", recall_score(labels_validation, xgb_predicted_labels))

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)
print("Neural Network:", recall_score(y_test, y_pred))

Recall of the models:
Logistic regression: 0.6428571428571429
Decision Tree: 0.6285714285714286
Naive Bayes: 0.6857142857142857
Support Vector Machine: 0.2714285714285714
Random Forest: 0.6714285714285714
Gradient boosting: 0.6857142857142857
AdaBoost: 0.6142857142857143
XGBoost: 0.7142857142857143
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step
Neural Network: 0.6626506024096386


#### Precision

In [113]:
from sklearn.metrics import precision_score

print("Precision of the models:")

lr_predicted_labels = lr_model.predict(features_validation)
print("Logistic regression:", precision_score(labels_validation, lr_predicted_labels))

dt_predicted_labels = dt_model.predict(features_validation)
print("Decision Tree:", precision_score(labels_validation, dt_predicted_labels))

nb_predicted_labels = nb_model.predict(features_validation)
print("Naive Bayes:", precision_score(labels_validation, nb_predicted_labels))

svm_predicted_labels = svm_model.predict(features_validation)
print("Support Vector Machine:", precision_score(labels_validation, svm_predicted_labels))

rf_predicted_labels = rf_model.predict(features_validation)
print("Random Forest:", precision_score(labels_validation, rf_predicted_labels))

gb_predicted_labels = gb_model.predict(features_validation)
print("Gradient boosting:", precision_score(labels_validation, gb_predicted_labels))

ab_predicted_labels = ab_model.predict(features_validation)
print("AdaBoost:", precision_score(labels_validation, ab_predicted_labels))

xgb_predicted_labels = xgboost_model.predict(features_validation_xgboost)
print("XGBoost:", precision_score(labels_validation, xgb_predicted_labels))

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)
print("Neural Network:", precision_score(y_test, y_pred))

Precision of the models:
Logistic regression: 0.7377049180327869
Decision Tree: 0.7333333333333333
Naive Bayes: 0.676056338028169
Support Vector Machine: 0.76
Random Forest: 0.734375
Gradient boosting: 0.8
AdaBoost: 0.6825396825396826
XGBoost: 0.7246376811594203
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
Neural Network: 0.859375


#### F1-score

In [114]:
from sklearn.metrics import f1_score

print("F1-scores of the models:")

lr_predicted_labels = lr_model.predict(features_validation)
print("Logistic regression:", f1_score(labels_validation, lr_predicted_labels))

dt_predicted_labels = dt_model.predict(features_validation)
print("Decision Tree:", f1_score(labels_validation, dt_predicted_labels))

nb_predicted_labels = nb_model.predict(features_validation)
print("Naive Bayes:", f1_score(labels_validation, nb_predicted_labels))

svm_predicted_labels = svm_model.predict(features_validation)
print("Support Vector Machine:", f1_score(labels_validation, svm_predicted_labels))

rf_predicted_labels = rf_model.predict(features_validation)
print("Random Forest:", f1_score(labels_validation, rf_predicted_labels))

gb_predicted_labels = gb_model.predict(features_validation)
print("Gradient boosting:", f1_score(labels_validation, gb_predicted_labels))

ab_predicted_labels = ab_model.predict(features_validation)
print("AdaBoost:", f1_score(labels_validation, ab_predicted_labels))

xgb_predicted_labels = xgboost_model.predict(features_validation_xgboost)
print("XGBoost:", f1_score(labels_validation, xgb_predicted_labels))

y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)
print("Neural Network:", f1_score(y_test, y_pred))

F1-scores of the models:
Logistic regression: 0.6870229007633588
Decision Tree: 0.676923076923077
Naive Bayes: 0.6808510638297872
Support Vector Machine: 0.4
Random Forest: 0.7014925373134329
Gradient boosting: 0.7384615384615385
AdaBoost: 0.6466165413533834
XGBoost: 0.7194244604316546
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
Neural Network: 0.7482993197278912


### 13.4.4 Testing the model

Finding the accuracy and the F1-score of the model in the testing set.

In [115]:
gb_model.score(features_test, labels_test)

0.8324022346368715

In [116]:
gb_predicted_test_labels = gb_model.predict(features_test)
f1_score(labels_test, gb_predicted_test_labels)

0.8026315789473685

## 13.5 Grid search

In [117]:
# Grid search with an rbf kernel

print("SVM grid search with a radial basis function kernel")

# rbf, C=1, gamma=0.1
svm_1_01 = SVC(kernel='rbf', C=1, gamma=0.1)
svm_1_01.fit(features_train, labels_train)
print("C=1, gamma=0.1", svm_1_01.score(features_validation, labels_validation))

# rbf, C=1, gamma=1
svm_1_1 = SVC(kernel='rbf', C=1, gamma=1)
svm_1_1.fit(features_train, labels_train)
print("C=1, gamma=1", svm_1_1.score(features_validation, labels_validation))

# rbf, C=1, gamma=10
svm_1_10 = SVC(kernel='rbf', C=1, gamma=10)
svm_1_10.fit(features_train, labels_train)
print("C=1, gamma=10", svm_1_10.score(features_validation, labels_validation))

# rbf, C=10, gamma=0.1
svm_10_01 = SVC(kernel='rbf', C=10, gamma=0.1)
svm_10_01.fit(features_train, labels_train)
print("C=10, gamma=0.1", svm_10_01.score(features_validation, labels_validation))

# rbf, C=10, gamma=1
svm_10_1 = SVC(kernel='rbf', C=10, gamma=1)
svm_10_1.fit(features_train, labels_train)
print("C=10, gamma=1", svm_10_1.score(features_validation, labels_validation))

# rbf, C=10, gamma=10
svm_10_10 = SVC(kernel='rbf', C=10, gamma=10)
svm_10_10.fit(features_train, labels_train)
print("C=10, gamma=10", svm_10_10.score(features_validation, labels_validation))

SVM grid search with a radial basis function kernel
C=1, gamma=0.1 0.702247191011236
C=1, gamma=1 0.6966292134831461
C=1, gamma=10 0.6685393258426966
C=10, gamma=0.1 0.7247191011235955
C=10, gamma=1 0.6910112359550562
C=10, gamma=10 0.651685393258427


In [118]:
from sklearn.model_selection import GridSearchCV

In [119]:
svm_parameters = {'kernel': ['rbf'],
                  'C': [0.01, 0.1, 1 , 10, 100],
                  'gamma': [0.01, 0.1, 1, 10, 100]
                }
svm = SVC()
svm_gs = GridSearchCV(estimator = svm,
                      param_grid = svm_parameters)
svm_gs.fit(features_train, labels_train)

svm_winner = svm_gs.best_estimator_
svm_winner

svm_winner.score(features_validation, labels_validation)

0.7191011235955056

In [120]:
svm_winner

SVC(C=10, gamma=0.01)

### 13.5.2 Grid search for XGboost model

In [121]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Prepare data
X_train = features_train.values
X_val   = features_validation.values
X_test  = features_test.values

y_train = labels_train.values
y_val   = labels_validation.values
y_test  = labels_test.values

# Base model
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=100
)

# Parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0]
}

# Grid search
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

# Results
print("Best parameters found:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)

# Test set evaluation
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("\nTest accuracy:", acc)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best parameters found: {'colsample_bytree': 0.7, 'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 200, 'subsample': 1.0}
Best cross-validation accuracy: 0.8332569211779228

Test accuracy: 0.8379888268156425

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.95      0.86        96
           1       0.92      0.71      0.80        83

    accuracy                           0.84       179
   macro avg       0.86      0.83      0.83       179
weighted avg       0.85      0.84      0.83       179



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [18:57:25] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## 13.6 Cross validation

In [125]:
# XGboost grid search params
grid.cv_results_

{'mean_fit_time': array([0.2322226 , 0.0354104 , 0.04047699, 0.07659836, 0.06765752,
        0.08407192, 0.24719605, 0.29496503, 0.12884827, 0.04748273,
        0.047824  , 0.03986154, 0.08425546, 0.08573794, 0.05787897,
        0.09749818, 0.126823  , 0.11251211, 0.05155454, 0.06956334,
        0.0561295 , 0.05935965, 0.06870604, 0.06428552, 0.10953455,
        0.09966578, 0.12967329, 0.05574718, 0.04064488, 0.0477984 ,
        0.05854378, 0.06281371, 0.0839191 , 0.08177977, 0.08194885,
        0.08662071, 0.01683006, 0.01785917, 0.0187304 , 0.03005352,
        0.02773967, 0.02954664, 0.05508757, 0.05194035, 0.04723353,
        0.0192585 , 0.02249241, 0.02081547, 0.03135786, 0.03324146,
        0.03068576, 0.05907693, 0.05929594, 0.06346574, 0.0258698 ,
        0.02199583, 0.02226357, 0.03610177, 0.03883672, 0.03618197,
        0.06530867, 0.07101383, 0.06915178, 0.02425394, 0.02716751,
        0.0242465 , 0.03992987, 0.04249597, 0.03998756, 0.07626152,
        0.07846727, 0.07827668,

In [122]:
svm_gs.cv_results_

{'mean_fit_time': array([0.00728006, 0.00750895, 0.00864244, 0.00890589, 0.00833611,
        0.00760994, 0.00778251, 0.00867577, 0.0091938 , 0.00857806,
        0.00730915, 0.00829787, 0.00881834, 0.01006265, 0.00987768,
        0.00939074, 0.00930252, 0.01095066, 0.01055713, 0.00933113,
        0.01131492, 0.01372323, 0.01002059, 0.01038547, 0.00955777]),
 'std_fit_time': array([5.59697136e-04, 4.83698123e-05, 6.49428210e-04, 1.83383041e-04,
        1.65968542e-04, 1.07290257e-03, 1.26527767e-04, 1.86257282e-04,
        1.76084451e-04, 1.25651070e-04, 2.32213890e-04, 3.08579942e-04,
        1.66739898e-04, 1.50506893e-03, 2.24360100e-03, 1.26994830e-03,
        6.49550801e-04, 1.18172477e-03, 4.06506499e-04, 7.63223416e-05,
        1.24590563e-03, 1.58387534e-03, 5.07380003e-04, 4.61600472e-04,
        3.30688938e-04]),
 'mean_score_time': array([0.00326509, 0.00333018, 0.00361643, 0.00375609, 0.00378284,
        0.00323462, 0.00336618, 0.00362186, 0.00375748, 0.00370498,
        0.00

## Exercise 13.1

### Loading and exploring the test dataset

In [139]:
# IMPORTANT: ONLY RUN THIS CELL IF YOU ARE WORKING ON A COLAB

url = "https://raw.githubusercontent.com/luisguiserrano/manning/master/Chapter_13_End_to_end_example/test.csv"
test_data = pd.read_csv(url)
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


### Cleaning up the test data

In [140]:
test_data.isna().sum()

,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Age,86
SibSp,0
Parch,0
Ticket,0
Fare,1
Cabin,327


#### Droping column that is missing too many values

In [141]:
# Cleaning the data
processed_test_data = test_data.drop('Cabin', axis=1)

#### Filling in missing data

Filling in missing ages with median age of Passengers

In [142]:
median_age = test_data["Age"].median()
median_age

processed_test_data["Age"] = processed_test_data["Age"].fillna(median_age)
median_age

27.0

In [143]:
processed_test_data.isna().sum()

,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,1
Embarked,0


Filling in missing Fares with mean (Average) fare of Passengers

In [144]:
average_fare = float(test_data["Fare"].mean())
processed_test_data["Fare"] = processed_test_data["Fare"].fillna(average_fare)
average_fare

35.627188489208635

In [146]:
processed_test_data.isna().sum()

,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0
Embarked,0


In [148]:
processed_test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,S


In [149]:
processed_test_data.to_csv('./cleaned_titanic_test_data.csv', index=None)

### Manipulating the features

In [185]:
processed_test_data = pd.read_csv('./cleaned_titanic_test_data.csv')
processed_test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,S


In [186]:
gender_columns = pd.get_dummies(processed_test_data['Sex'], prefix='Sex')
embarked_columns = pd.get_dummies(processed_test_data["Embarked"], prefix="Embarked")

processed_test_data = pd.concat([processed_test_data, gender_columns], axis=1)
processed_test_data = pd.concat([processed_test_data, embarked_columns], axis=1)
processed_test_data = processed_test_data.assign(Embarked_U=False)

processed_test_data = processed_test_data.drop(['Sex', 'Embarked'], axis=1)

In [187]:
class_survived = processed_test_data[['Pclass']]

first_class = class_survived[class_survived['Pclass'] == 1]
second_class = class_survived[class_survived['Pclass'] == 2]
third_class = class_survived[class_survived['Pclass'] == 3]

In [188]:
categorized_pclass_columns = pd.get_dummies(processed_test_data['Pclass'], prefix='Pclass')
processed_test_data = pd.concat([processed_test_data, categorized_pclass_columns], axis=1)
processed_test_data = processed_test_data.drop(['Pclass'], axis=1)

In [190]:
processed_test_data.head()

,PassengerId,Name,Age,SibSp,Parch,Ticket,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3
0,892,"Kelly, Mr. James",34.5,0,0,330911,7.8292,False,True,False,True,False,False,False,False,True
1,893,"Wilkes, Mrs. James (Ellen Needs)",47.0,1,0,363272,7.0000,True,False,False,False,True,False,False,False,True
2,894,"Myles, Mr. Thomas Francis",62.0,0,0,240276,9.6875,False,True,False,True,False,False,False,True,False
3,895,"Wirz, Mr. Albert",27.0,0,0,315154,8.6625,False,True,False,False,True,False,False,False,True
4,896,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",22.0,1,1,3101298,12.2875,True,False,False,False,True,False,False,False,True


#### Binning

In [191]:
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
categorized_age_test = pd.cut(processed_test_data['Age'], bins)
processed_test_data['Categorized_age'] = categorized_age_test
processed_test_data = processed_test_data.drop(["Age"], axis=1)

In [192]:
cagegorized_age_columns_test = pd.get_dummies(processed_test_data['Categorized_age'], prefix='Categorized_age')
processed_test_data = pd.concat([processed_test_data, cagegorized_age_columns_test], axis=1)
processed_test_data = processed_test_data.drop(['Categorized_age'], axis=1)

#### Feature selection

In [193]:
processed_test_data = processed_test_data.drop(['Name', 'Ticket', 'PassengerId'], axis=1)

In [194]:
processed_test_data.head()

,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Embarked_U,Pclass_1,Pclass_2,Pclass_3,"Categorized_age_(0, 10]","Categorized_age_(10, 20]","Categorized_age_(20, 30]","Categorized_age_(30, 40]","Categorized_age_(40, 50]","Categorized_age_(50, 60]","Categorized_age_(60, 70]","Categorized_age_(70, 80]"
0,0,0,7.8292,False,True,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False
1,1,0,7.0000,True,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False
2,0,0,9.6875,False,True,False,True,False,False,False,True,False,False,False,False,False,False,False,True,False
3,0,0,8.6625,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False
4,1,1,12.2875,True,False,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False


In [195]:
processed_test_data.to_csv('./preprocessed_titanic_test_data.csv', index=None)

### Testing the model

In [196]:
# XGboost grid searched (hyperparam tuned) model
best_model.predict(processed_test_data)

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0,
       1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [206]:
print("From", len(processed_test_data), "Passengers", best_model.predict(processed_test_data).sum(), "survived according to XGboost (best) model")

From 418 Passengers 125 survived according to XGboost (best) model


In [205]:
lr_model.predict(processed_test_data)

print("What other models think, how many passengers survived Titanic from test set:")
print("Logistic regression: Total:", len(processed_test_data), "Survived:", lr_model.predict(processed_test_data).sum())
print("Decision tree: Total:", len(processed_test_data), "Survived:", dt_model.predict(processed_test_data).sum())
print("Naive Bayes: Total:", len(processed_test_data), "Survived:", nb_model.predict(processed_test_data).sum())
print("SVM: Total:", len(processed_test_data), "Survived:", svm_model.predict(processed_test_data).sum())
print("SVM (winner): Total:", len(processed_test_data), "Survived:", svm_winner.predict(processed_test_data).sum())
print("Random forest: Total:", len(processed_test_data), "Survived:", rf_model.predict(processed_test_data).sum())
print("Gradient boosting: Total:", len(processed_test_data), "Survived:", gb_model.predict(processed_test_data).sum())
print("AdaBoost: Total:", len(processed_test_data), "Survived:", ab_model.predict(processed_test_data).sum())

processed_test_data_xgboost = processed_test_data.rename(columns=clean_col)
print("XGboost: Total:", len(processed_test_data), "Survived:", xgboost_model.predict(processed_test_data_xgboost).sum())

processed_test_data_float  = features_test.values.astype("float32")
y_pred_proba_nn = model.predict(processed_test_data_float)
y_pred_nn = (y_pred_proba > 0.5).astype(int)
print("Neural Network: Total:", len(processed_test_data), "Survived:", len(y_pred_nn))

What other models think, how many passengers survived Titanic from test set:
Logistic regression: Total: 418 Survived: 142
Decision tree: Total: 418 Survived: 140
Naive Bayes: Total: 418 Survived: 182
SVM: Total: 418 Survived: 61
SVM (winner): Total: 418 Survived: 151
Random forest: Total: 418 Survived: 140
Gradient boosting: Total: 418 Survived: 150
AdaBoost: Total: 418 Survived: 156
XGboost: Total: 418 Survived: 138
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Neural Network: Total: 418 Survived: 179


I think from test set around 120-150 people survived on average